# Silver — CRM Customer Info
Customer master data from the CRM.

`bronze.crm_cust_info` → `silver.crm_customers`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_cust_info")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Normalize coded values
Single-letter codes become readable labels; unknown values become `n/a` so downstream joins/filters are predictable.

In [ ]:
df = normalize(df, "cst_marital_status", {"S": "Single", "M": "Married"})
df = normalize(df, "cst_gndr", {"F": "Female", "M": "Male"})

### Drop records without a customer id
A customer without an id cannot be joined to anything.

In [ ]:
df = df.filter(col("cst_id").isNotNull())

### Keep the latest record per customer
The CRM exports some customers twice (an older incomplete row and a newer complete one). Keep the most recent by `cst_create_date`.

In [ ]:
latest = Window.partitionBy("cst_id").orderBy(col("cst_create_date").desc())
df = df.withColumn("_rn", F.row_number().over(latest)).filter(col("_rn") == 1).drop("_rn")

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.crm_customers")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10;